In [ ]:
pip install -U ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
USDA_API_KEY = userdata.get('USDA_API_KEY')
# Stored as a secret on my colab, everyone using this needs their own.

In [ ]:
# Putting all imports here just for convenience, it gets repeated later anyway
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm
from ultralytics import YOLO
import os
import cv2
import numpy as np
from PIL import Image, ImageOps
from IPython.display import Image, display
import os
import cv2
import numpy as np
import requests
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO


In [ ]:
%cd /content/

# To copy raw dataset zip file from Drive to local environment
!cp /content/drive/MyDrive/foodseg/foodseg.zip .

!unzip -o /content/foodseg.zip -d /content/

In [ ]:
import os
import cv2
import numpy as np
import shutil
from tqdm import tqdm

SOURCE_BASE_DIR = r"/content/FoodSeg103"
TARGET_OUTPUT_DIR = r"/content/yolo_foodseg"

TASK_SEGMENTATION = True
MIN_CONTOUR_AREA = 30   # min area for detection
EPSILON_FACTOR = 0.003  # for polygon making ( this is very sharp )

def convert_split(source_split_name, target_split_name):
    img_source_folder = os.path.join(SOURCE_BASE_DIR, "Images", "img_dir", source_split_name)
    mask_source_folder = os.path.join(SOURCE_BASE_DIR, "Images", "ann_dir", source_split_name)

    img_target_folder = os.path.join(TARGET_OUTPUT_DIR, "images", target_split_name)
    lbl_target_folder = os.path.join(TARGET_OUTPUT_DIR, "labels", target_split_name)

    os.makedirs(img_target_folder, exist_ok=True)
    os.makedirs(lbl_target_folder, exist_ok=True)

    if not os.path.exists(mask_source_folder):
        print(f"Directory missing: {mask_source_folder}")
        return

    mask_files = [f for f in os.listdir(mask_source_folder) if f.endswith(('.png', '.jpg', '.jpeg'))]
    print(f"\nProcessing '{source_split_name}' -> Mapping to YOLO '{target_split_name}'...")

    stats = {"no_image": 0, "no_polygons": 0, "total_polygons": 0, "empty_labels": 0}

    for filename in tqdm(mask_files):
        base_name = os.path.splitext(filename)[0]
        mask_path = os.path.join(mask_source_folder, filename)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        h, w = mask.shape

        img_copied = False
        for ext in ['.jpg', '.png', '.jpeg']:
            possible_img_path = os.path.join(img_source_folder, base_name + ext)
            if os.path.exists(possible_img_path):
                shutil.copy(possible_img_path, os.path.join(img_target_folder, base_name + ext))
                img_copied = True
                break

        if not img_copied:
            stats["no_image"] += 1
            continue

        unique_classes = np.unique(mask)
        yolo_lines = []

        for class_id in unique_classes:
            if class_id == 0:
                continue

            class_mask = np.where(mask == class_id, 255, 0).astype(np.uint8)
            contours, _ = cv2.findContours(class_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            yolo_class_idx = int(class_id) - 1

            for contour in contours:
                if cv2.contourArea(contour) < MIN_CONTOUR_AREA:
                    continue

                if TASK_SEGMENTATION:
                    # to simplify polygon and avoid bloated label files
                    epsilon = EPSILON_FACTOR * cv2.arcLength(contour, True)
                    simplified = cv2.approxPolyDP(contour, epsilon, True)

                    if len(simplified) < 3:
                        continue  # not a valid polygon after simplification

                    normalized_coords = []
                    for point in simplified:
                        pt_x, pt_y = point[0]
                        nx = min(max(pt_x / w, 0.0), 1.0)
                        ny = min(max(pt_y / h, 0.0), 1.0)
                        normalized_coords.append(f"{nx:.6f} {ny:.6f}")

                    yolo_lines.append(f"{yolo_class_idx} " + " ".join(normalized_coords))
                    stats["total_polygons"] += 1
                else:
                    x, y, box_w, box_h = cv2.boundingRect(contour)
                    cx = min(max((x + box_w / 2.0) / w, 0.0), 1.0)
                    cy = min(max((y + box_h / 2.0) / h, 0.0), 1.0)
                    yolo_lines.append(f"{yolo_class_idx} {cx:.6f} {cy:.6f} {box_w/w:.6f} {box_h/h:.6f}")
                    stats["total_polygons"] += 1

        if not yolo_lines:
            stats["empty_labels"] += 1

        with open(os.path.join(lbl_target_folder, f"{base_name}.txt"), 'w') as f:
            f.write("\n".join(yolo_lines))

    print(f"  Images missing pairs: {stats['no_image']}")
    print(f"  Empty label files (no valid polygons): {stats['empty_labels']}")
    print(f"  Total polygons written: {stats['total_polygons']}")

convert_split('train', 'train')
convert_split('test', 'val')
print("\nConversion complete! Dataset ready inside /content/yolo_foodseg")

In [ ]:
# Creating the .yaml file for model training
%%writefile /content/yolo_foodseg/data.yaml
path: /content/yolo_foodseg # root directory in colab
train: images/train
val: images/val
nc: 103
names:
  0: candy
  1: egg tart
  2: french fries
  3: chocolate
  4: biscuit
  5: popcorn
  6: pudding
  7: ice cream
  8: cheese butter
  9: cake
  10: wine
  11: milkshake
  12: coffee
  13: juice
  14: milk
  15: tea
  16: almond
  17: red beans
  18: cashew
  19: dried cranberries
  20: soy
  21: walnut
  22: peanut
  23: egg
  24: apple
  25: date
  26: apricot
  27: avocado
  28: banana
  29: strawberry
  30: cherry
  31: blueberry
  32: raspberry
  33: mango
  34: olives
  35: peach
  36: lemon
  37: pear
  38: fig
  39: pineapple
  40: grape
  41: kiwi
  42: melon
  43: orange
  44: watermelon
  45: steak
  46: pork
  47: chicken duck
  48: sausage
  49: fried meat
  50: lamb
  51: sauce
  52: crab
  53: fish
  54: shellfish
  55: shrimp
  56: soup
  57: bread
  58: corn
  59: hamburg
  60: pizza
  61: hanamaki baozi
  62: wonton dumplings
  63: pasta
  64: noodles
  65: rice
  66: pie
  67: tofu
  68: eggplant
  69: potato
  70: garlic
  71: cauliflower
  72: tomato
  73: kelp
  74: seaweed
  75: spring onion
  76: rape
  77: ginger
  78: okra
  79: lettuce
  80: pumpkin
  81: cucumber
  82: white radish
  83: carrot
  84: asparagus
  85: bamboo shoots
  86: broccoli
  87: celery stick
  88: cilantro mint
  89: snow peas
  90: cabbage
  91: bean sprouts
  92: onion
  93: pepper
  94: green beans
  95: French beans
  96: king oyster mushroom
  97: shiitake
  98: enoki mushroom
  99: oyster mushroom
  100: white button mushroom
  101: salad
  102: other ingredients

In [ ]:
import os
import shutil
from ultralytics import YOLO


# TRAINING MODE : ( purely for my convenience )
# 1) "scratch" : train a fresh yolov8n-seg.pt from epoch 0
# 2) "checkpoint" : fine-tune from a saved .pt (weights only), NEW optimizer
#                   /LR schedule; to be used when you only bare weights are present.
# 3) "resume" : Resume of an interrupted run using its own last.pt
#               (must still be inside its original run folder next to args.yaml)

TRAIN_MODE = "checkpoint"

DATA_YAML = "/content/yolo_foodseg/data.yaml"
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/foodseg/checkpoints"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

PROJECT_DIR = "/content/runs"


def backup_checkpoint_to_drive(trainer):
    # To run after every epoch; copies last.pt/best.pt to Drive every 5 epochs.
    epoch_num = trainer.epoch + 1
    if epoch_num % 5 == 0:
        weights_dir = trainer.save_dir / "weights"
        for fname in ["last.pt", "best.pt"]:
            src = weights_dir / fname
            if src.exists():
                dst = os.path.join(DRIVE_CHECKPOINT_DIR, f"epoch{epoch_num}_{fname}")
                shutil.copy(src, dst)
        print(f"[Checkpoint backup] Copied last.pt/best.pt to Drive at epoch {epoch_num}")


if TRAIN_MODE == "scratch":
    model = YOLO("yolov8n-seg.pt")
    model.add_callback("on_train_epoch_end", backup_checkpoint_to_drive)

    results = model.train(
        data=DATA_YAML,
        epochs=50,
        imgsz=640,
        batch=16,
        device=0,
        project=PROJECT_DIR,
        name="foodseg103_run1",
        save_period=5,
    )

elif TRAIN_MODE == "checkpoint":
    # Fine-tune from a saved checkpoint (weights only)
    # Use this when you only have the .pt file itself (e.g. epoch35_last.pt
    # copied to Drive) and NOT the original run folder with args.yaml.
    # This starts a fresh optimizer + LR schedule, so keep the LR low and
    # skip warmup to avoid the loss spiking/plateauing from a hard restart.
    CHECKPOINT_PATH = "/content/drive/MyDrive/foodseg/final_run/weights/last.pt"
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Could not find checkpoint at {CHECKPOINT_PATH}.")

    model = YOLO(CHECKPOINT_PATH)
    model.add_callback("on_train_epoch_end", backup_checkpoint_to_drive)

    results = model.train(
        data=DATA_YAML,
        epochs=60,
        imgsz=640,
        batch=16,
        device=0,
        project=PROJECT_DIR,
        name="foodseg103_run1_finetune",
        save_period=5,
    )

elif TRAIN_MODE == "resume":
    # True resume from last.pt
    # This ONLY works if RESUME_RUN_DIR still contains last.pt alongside its
    # original args.yaml (i.e. the actual run folder from Ultralytics, not a
    # bare weights file copied elsewhere). resume=True restores the optimizer
    # state and continues the ORIGINAL LR schedule exactly where it left off.
    RESUME_RUN_DIR = os.path.join(PROJECT_DIR, "foodseg103_run1")
    RESUME_WEIGHTS = os.path.join(RESUME_RUN_DIR, "weights", "last.pt")
    RESUME_ARGS = os.path.join(RESUME_RUN_DIR, "args.yaml")

    if not os.path.exists(RESUME_WEIGHTS) or not os.path.exists(RESUME_ARGS):
        raise FileNotFoundError(
            f"True resume requires BOTH {RESUME_WEIGHTS} and {RESUME_ARGS} to exist "
            f"in the original Ultralytics run folder. A last.pt copied elsewhere "
            f"(e.g. Drive's final_run/weights/last.pt) is NOT enough, since args.yaml "
            f"won't be next to it there. If your Colab runtime was reset since training, "
            f"this folder is gone and true resume isn't possible — use "
            f"TRAIN_MODE='checkpoint' instead, pointed at the Drive copy of last.pt."
        )

    model = YOLO(RESUME_WEIGHTS)
    model.add_callback("on_train_epoch_end", backup_checkpoint_to_drive)

    # No other train() args needed/allowed here as resume=True reads the
    # original run's args.yaml (data, epochs, imgsz, batch, etc.) automatically.
    results = model.train(resume=True)

else:
    raise ValueError(f"Unknown TRAIN_MODE: {TRAIN_MODE!r}")

RUN_DIR = str(results.save_dir)
print(f"\nTraining complete. Local run directory: {RUN_DIR}")


In [ ]:
import os
import glob
import shutil

# Final destination on Drive for the fully-trained model + its final diagnostic images
DRIVE_FINAL_DIR = "/content/drive/MyDrive/foodseg/final_run"
os.makedirs(os.path.join(DRIVE_FINAL_DIR, "weights"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_FINAL_DIR, "images"), exist_ok=True)

# Final weights
for fname in ["best.pt", "last.pt"]:
    src = os.path.join(RUN_DIR, "weights", fname)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_FINAL_DIR, "weights", fname))
        print(f"Saved {fname} to Drive.")
    else:
        print(f"WARNING: {fname} not found at {src}")

# Final iteration of train/val batch images + summary plots logged during training
# Ultralytics overwrites these each epoch, so whatever's on disk now IS the final iteration)
image_patterns = [
    "train_batch*.jpg", "val_batch*.jpg",
    "results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
    "PR_curve.png", "F1_curve.png", "labels.jpg"
]
copied = 0
for pattern in image_patterns:
    for img_path in glob.glob(os.path.join(RUN_DIR, pattern)):
        shutil.copy(img_path, os.path.join(DRIVE_FINAL_DIR, "images", os.path.basename(img_path)))
        copied += 1

print(f"Copied {copied} image(s) to Drive.")
print(f"\nAll final weights + images backed up to: {DRIVE_FINAL_DIR}")


In [ ]:
from IPython.display import Image, display
import os

latest_run = RUN_DIR if 'RUN_DIR' in dir() else '/content/runs/foodseg103_run1_finetune'

print("--- Ground Truth Labels (What the dataset looks like) ---")
labels_path = os.path.join(latest_run, 'val_batch2_labels.jpg')
if os.path.exists(labels_path):
    display(Image(filename=labels_path, width=800))

print("\n--- Model Predictions (final epoch) ---")
pred_path = os.path.join(latest_run, 'val_batch2_pred.jpg')
if os.path.exists(pred_path):
    display(Image(filename=pred_path, width=800))


In [ ]:
results_path = os.path.join(latest_run, 'results.png')

if os.path.exists(results_path):
    print("--- Training Metrics Plot ---")
    display(Image(filename=results_path, width=1000))
else:
    print("results.png not found. Ensure the training cell finished running without errors.")


In [ ]:
# resizing/letterboxing/normalization is handled intenally
# EXIF rotation from phone cameras, and non-RGB modes handling :

import os
import cv2
import numpy as np
from PIL import Image, ImageOps


def load_and_preprocess_image(image_path: str) -> np.ndarray:
    """
    Loads an image from disk and returns a clean BGR numpy array (OpenCV/YOLO format).
    - Corrects EXIF orientation (common issue with photos taken on phones).
    - Converts any color mode (RGBA, grayscale, palette) to standard RGB -> BGR.
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found at: {image_path}")

    pil_img = Image.open(image_path)
    pil_img = ImageOps.exif_transpose(pil_img)   # fix rotated phone photos
    pil_img = pil_img.convert("RGB")             # normalize color mode

    rgb_array = np.array(pil_img)
    bgr_array = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)

    h, w = bgr_array.shape[:2]
    print(f"Loaded image: {os.path.basename(image_path)} | size: {w}x{h}")
    return bgr_array


In [ ]:
FOOD_METRICS = {
    "candy": {"type": "countable", "avg_weight": 10, "fdc_id": "169643"},
    "egg tart": {"type": "countable", "avg_weight": 60, "fdc_id": None},
    "french fries": {"type": "area", "density_factor": 0.35, "fdc_id": "170002"},
    "chocolate": {"type": "area", "density_factor": 0.9, "fdc_id": "170271"},
    "biscuit": {"type": "countable", "avg_weight": 15, "fdc_id": None},
    "popcorn": {"type": "area", "density_factor": 0.05, "fdc_id": "169695"},
    "pudding": {"type": "area", "density_factor": 0.9, "fdc_id": None},
    "ice cream": {"type": "area", "density_factor": 0.6, "fdc_id": "172421"},
    "cheese butter": {"type": "area", "density_factor": 0.8, "fdc_id": "173414"},
    "cake": {"type": "countable", "avg_weight": 120, "fdc_id": "172900"},
    "wine": {"type": "container", "avg_depth_cm": 5.0, "density_g_cm3": 0.99, "fdc_id": "174837"},
    "milkshake": {"type": "container", "avg_depth_cm": 12.0, "density_g_cm3": 1.03, "fdc_id": None},
    "coffee": {"type": "container", "avg_depth_cm": 8.0, "density_g_cm3": 1.0, "fdc_id": "171890"},
    "juice": {"type": "container", "avg_depth_cm": 10.0, "density_g_cm3": 1.03, "fdc_id": "168155"},
    "milk": {"type": "container", "avg_depth_cm": 10.0, "density_g_cm3": 1.03, "fdc_id": "718921"},
    "tea": {"type": "container", "avg_depth_cm": 8.0, "density_g_cm3": 1.0, "fdc_id": None},
    "almond": {"type": "area", "density_factor": 0.4, "fdc_id": "170567"},
    "red beans": {"type": "area", "density_factor": 0.7, "fdc_id": None},
    "cashew": {"type": "area", "density_factor": 0.4, "fdc_id": "170162"},
    "dried cranberries": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "soy": {"type": "area", "density_factor": 0.7, "fdc_id": None},
    "walnut": {"type": "area", "density_factor": 0.35, "fdc_id": "170187"},
    "peanut": {"type": "area", "density_factor": 0.4, "fdc_id": "172430"},
    "egg": {"type": "countable", "avg_weight": 50, "fdc_id": "171287"},
    "apple": {"type": "countable", "avg_weight": 182, "fdc_id": "1750337"},
    "date": {"type": "countable", "avg_weight": 8, "fdc_id": None},
    "apricot": {"type": "countable", "avg_weight": 35, "fdc_id": None},
    "avocado": {"type": "countable", "avg_weight": 150, "fdc_id": "171705"},
    "banana": {"type": "countable", "avg_weight": 118, "fdc_id": "1105314"},
    "strawberry": {"type": "countable", "avg_weight": 12, "fdc_id": "167762"},
    "cherry": {"type": "countable", "avg_weight": 8, "fdc_id": "171719"},
    "blueberry": {"type": "countable", "avg_weight": 1, "fdc_id": "171711"},
    "raspberry": {"type": "countable", "avg_weight": 4, "fdc_id": None},
    "mango": {"type": "countable", "avg_weight": 200, "fdc_id": None},
    "olives": {"type": "countable", "avg_weight": 4, "fdc_id": "169300"},
    "peach": {"type": "countable", "avg_weight": 150, "fdc_id": None},
    "lemon": {"type": "countable", "avg_weight": 60, "fdc_id": "167746"},
    "pear": {"type": "countable", "avg_weight": 178, "fdc_id": "169118"},
    "fig": {"type": "countable", "avg_weight": 40, "fdc_id": None},
    "pineapple": {"type": "area", "density_factor": 0.5, "fdc_id": "169124"},
    "grape": {"type": "countable", "avg_weight": 5, "fdc_id": "174683"},
    "kiwi": {"type": "countable", "avg_weight": 75, "fdc_id": "168153"},
    "melon": {"type": "area", "density_factor": 0.5, "fdc_id": "169094"},
    "orange": {"type": "countable", "avg_weight": 131, "fdc_id": None},
    "watermelon": {"type": "area", "density_factor": 0.55, "fdc_id": "167765"},
    "steak": {"type": "countable", "avg_weight": 250, "fdc_id": "172207"},
    "pork": {"type": "countable", "avg_weight": 200, "fdc_id": None},
    "chicken duck": {"type": "countable", "avg_weight": 150, "fdc_id": None},
    "sausage": {"type": "countable", "avg_weight": 75, "fdc_id": "174268"},
    "fried meat": {"type": "countable", "avg_weight": 150, "fdc_id": None},
    "lamb": {"type": "countable", "avg_weight": 200, "fdc_id": None},
    "sauce": {"type": "area", "density_factor": 0.5, "fdc_id": "172447"},
    "crab": {"type": "countable", "avg_weight": 130, "fdc_id": "174241"},
    "fish": {"type": "countable", "avg_weight": 150, "fdc_id": "174220"},
    "shellfish": {"type": "countable", "avg_weight": 100, "fdc_id": None},
    "shrimp": {"type": "countable", "avg_weight": 15, "fdc_id": "175171"},
    "soup": {"type": "container", "avg_depth_cm": 6.0, "density_g_cm3": 1.02, "fdc_id": "171457"},
    "bread": {"type": "countable", "avg_weight": 30, "fdc_id": "172686"},
    "corn": {"type": "area", "density_factor": 0.5, "fdc_id": "169290"},
    "hamburg": {"type": "countable", "avg_weight": 220, "fdc_id": "174033"},
    "pizza": {"type": "area", "density_factor": 0.7, "fdc_id": "173292"},
    "hanamaki baozi": {"type": "countable", "avg_weight": 60, "fdc_id": None},
    "wonton dumplings": {"type": "countable", "avg_weight": 30, "fdc_id": "173322"},
    "pasta": {"type": "area", "density_factor": 0.6, "fdc_id": None},
    "noodles": {"type": "area", "density_factor": 0.6, "fdc_id": "169739"},
    "rice": {"type": "area", "density_factor": 0.5, "fdc_id": "169757"},
    "pie": {"type": "countable", "avg_weight": 135, "fdc_id": "173255"},
    "tofu": {"type": "area", "density_factor": 0.6, "fdc_id": None},
    "eggplant": {"type": "countable", "avg_weight": 300, "fdc_id": "169226"},
    "potato": {"type": "countable", "avg_weight": 150, "fdc_id": "170026"},
    "garlic": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "cauliflower": {"type": "area", "density_factor": 0.4, "fdc_id": "169975"},
    "tomato": {"type": "countable", "avg_weight": 120, "fdc_id": "170457"},
    "kelp": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "seaweed": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "spring onion": {"type": "area", "density_factor": 0.2, "fdc_id": None},
    "ginger": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "okra": {"type": "area", "density_factor": 0.4, "fdc_id": None},
    "lettuce": {"type": "area", "density_factor": 0.15, "fdc_id": None},
    "pumpkin": {"type": "area", "density_factor": 0.5, "fdc_id": "168448"},
    "cucumber": {"type": "area", "density_factor": 0.35, "fdc_id": "168409"},
    "white radish": {"type": "area", "density_factor": 0.3, "fdc_id": "169279"},
    "carrot": {"type": "countable", "avg_weight": 60, "fdc_id": "170393"},
    "asparagus": {"type": "area", "density_factor": 0.35, "fdc_id": "168387"},
    "bamboo shoots": {"type": "area", "density_factor": 0.35, "fdc_id": None},
    "broccoli": {"type": "area", "density_factor": 0.4, "fdc_id": "170379"},
    "celery stick": {"type": "area", "density_factor": 0.3, "fdc_id": "169234"},
    "cilantro mint": {"type": "area", "density_factor": 0.15, "fdc_id": "170428"},
    "snow peas": {"type": "area", "density_factor": 0.35, "fdc_id": "170419"},
    "cabbage": {"type": "area", "density_factor": 0.3, "fdc_id": "169228"},
    "bean sprouts": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "onion": {"type": "area", "density_factor": 0.35, "fdc_id": "170000"},
    "pepper": {"type": "countable", "avg_weight": 120, "fdc_id": "170427"},
    "green beans": {"type": "area", "density_factor": 0.35, "fdc_id": "169230"},
    "French beans": {"type": "area", "density_factor": 0.35, "fdc_id": None},
    "king oyster mushroom": {"type": "area", "density_factor": 0.35, "fdc_id": "169251"},
    "shiitake": {"type": "area", "density_factor": 0.35, "fdc_id": None},
    "enoki mushroom": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "oyster mushroom": {"type": "area", "density_factor": 0.3, "fdc_id": None},
    "white button mushroom": {"type": "area", "density_factor": 0.35, "fdc_id": "169251"},
    "salad": {"type": "area", "density_factor": 0.2, "fdc_id": "170425"},
    "other ingredients": {"type": "area", "density_factor": 0.4, "fdc_id": None},
}


In [ ]:
import cv2
import numpy as np
import requests
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO

class PlateCalc:
    def __init__(
        self,
        model_path: str,
        usda_api_key: str = None,
        conf_threshold: float = 0.35,
        plate_diameter_cm: float = 26.0,
    ):
        if model_path and os.path.exists(model_path):
            self.model = YOLO(model_path)
            print(f"Loaded custom weights from {model_path}")
        else:
            raise FileNotFoundError(
                f"Trained weights not found at '{model_path}'. "
                f"Run the training cells above first, or point model_path at your best.pt."
            )

        self.api_key = usda_api_key
        self.class_names = self.model.names          # {class_id: class_name}
        self.conf_threshold = conf_threshold
        self.plate_diameter_cm = plate_diameter_cm
        self._kcal_cache = {}                          # fdc_id/class_name -> kcal per 100g


    def _estimate_plate_scale(self, image: np.ndarray):
        """
        Finds the plate in the image and returns px_per_cm2, i.e. how many
        pixels correspond to one real-world cm^2 in THIS photo.

        Falls back to None if no plate-like circle can be found; callers
        should fall back to a conservative default in that case rather than
        silently mis-scaling every food item.
        """
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.medianBlur(gray, 7)
        h, w = gray.shape[:2]

        circles = cv2.HoughCircles(
            gray,
            cv2.HOUGH_GRADIENT,
            dp=1.2,
            minDist=min(h, w) // 2,
            param1=100,
            param2=60,
            minRadius=int(min(h, w) * 0.25),
            maxRadius=int(min(h, w) * 0.60),
        )

        if circles is None:
            print("  [calibration] No plate circle detected -- falling back to default scale.")
            return None

        # Take the largest detected circle as the plate.
        circles = np.round(circles[0, :]).astype(int)
        _, _, radius_px = max(circles, key=lambda c: c[2])
        plate_diameter_px = 2 * radius_px

        plate_diameter_cm = self.plate_diameter_cm
        px_per_cm = plate_diameter_px / plate_diameter_cm
        px_per_cm2 = px_per_cm ** 2

        print(
            f"  [calibration] Detected plate diameter: {plate_diameter_px}px "
            f"-> {px_per_cm:.2f} px/cm -> {px_per_cm2:.1f} px^2/cm^2"
        )
        return px_per_cm2

    def _fetch_kcal_by_fdc_id(self, fdc_id: str):
        url = f"https://api.nal.usda.gov/fdc/v1/food/{fdc_id}"
        try:
            resp = requests.get(url, params={"api_key": self.api_key}, timeout=6)
            if resp.status_code == 200:
                data = resp.json()
                for nutrient in data.get("foodNutrients", []):
                    info = nutrient.get("nutrient", {})
                    if info.get("unitName") == "KCAL" or "Energy" in info.get("name", ""):
                        if nutrient.get("amount") is not None:
                            return float(nutrient["amount"])
            else:
                print(f"  [USDA] fdc_id={fdc_id} returned HTTP {resp.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"  [USDA] request error for fdc_id={fdc_id}: {e}")
        return None

    def _fetch_kcal_by_search(self, class_name: str):
        url = "https://api.nal.usda.gov/fdc/v1/foods/search"
        try:
            resp = requests.get(url, params={
                "api_key": self.api_key,
                "query": class_name,
                "pageSize": 1,
                "dataType": ["Foundation", "SR Legacy"],
            }, timeout=6)
            if resp.status_code == 200:
                foods = resp.json().get("foods", [])
                if foods:
                    for nutrient in foods[0].get("foodNutrients", []):
                        name = nutrient.get("nutrientName", "")
                        if nutrient.get("unitName") == "KCAL" or "Energy" in name:
                            if nutrient.get("value") is not None:
                                return float(nutrient["value"])
        except requests.exceptions.RequestException as e:
            print(f"  [USDA] search error for '{class_name}': {e}")
        return None

    def get_kcal_per_100g(self, class_name: str, fdc_id: str) -> float:
        cache_key = fdc_id or class_name
        if cache_key in self._kcal_cache:
            return self._kcal_cache[cache_key]

        kcal = None
        if self.api_key:
            if fdc_id:
                kcal = self._fetch_kcal_by_fdc_id(fdc_id)
            if kcal is None:
                kcal = self._fetch_kcal_by_search(class_name)
        else:
            print("  [USDA] No API key configured -- skipping live lookup.")

        if kcal is None:
            print(f"  [fallback] Using generic 140 kcal/100g estimate for '{class_name}'.")
            kcal = 140.0

        self._kcal_cache[cache_key] = kcal
        return kcal


    def _estimate_class_weight(
        self,
        class_name: str,
        combined_mask: np.ndarray,
        instance_count_hint: int,
        px_per_cm2: float,
    ):
        metrics = FOOD_METRICS.get(class_name)
        if metrics is None:
            print(f"  [fallback] '{class_name}' not in FOOD_METRICS -- using generic flat-food area estimate.")
            metrics = {"type": "area", "density_factor": 0.4, "fdc_id": None}

        if metrics["type"] == "countable":
            num_labels, _, stats, _ = cv2.connectedComponentsWithStats(
                (combined_mask * 255).astype(np.uint8), connectivity=8
            )
            blob_count = sum(1 for i in range(1, num_labels) if stats[i, cv2.CC_STAT_AREA] >= 150)
            instance_count = max(blob_count, instance_count_hint, 1)
            weight_g = instance_count * metrics.get("avg_weight", 100)
            qty_desc = f"{instance_count} instance(s)"

        elif metrics["type"] == "container":
            # it takes the estimate volume as surface_area x assumed_depth, then
            # converts to weight via the liquid's density
            pixel_area = int(np.sum(combined_mask))
            surface_area_cm2 = pixel_area / px_per_cm2
            depth_cm = metrics.get("avg_depth_cm", 8.0)
            density_g_cm3 = metrics.get("density_g_cm3", 1.0)
            volume_cm3 = surface_area_cm2 * depth_cm
            weight_g = volume_cm3 * density_g_cm3
            qty_desc = f"{surface_area_cm2:.1f} cm^2 surface x {depth_cm:.1f} cm deep"

        else:  # "area" -- flat plated food (thin layer on the plate)
            pixel_area = int(np.sum(combined_mask))
            real_area_cm2 = pixel_area / px_per_cm2
            weight_g = real_area_cm2 * metrics.get("density_factor", 0.4)
            qty_desc = f"{real_area_cm2:.1f} cm^2 mask area"

        return weight_g, qty_desc, metrics.get("fdc_id")

    def analyze_plate(self, image_path: str, show_plot: bool = True) -> pd.DataFrame:
        image = load_and_preprocess_image(image_path)
        h, w = image.shape[:2]

        px_per_cm2 = self._estimate_plate_scale(image)
        if px_per_cm2 is None:
            # I assume the plate fills ~70% of the shorter
            # image dimension, rather than reusing a fixed g/px constant
            # tuned to a different, unknown reference photo.
            # this should average out the +/- of calories ( hopefully )
            assumed_plate_diameter_px = 0.7 * min(h, w)
            px_per_cm = assumed_plate_diameter_px / self.plate_diameter_cm
            px_per_cm2 = px_per_cm ** 2
            print(
                f"  [calibration] Using fallback scale based on assumed plate "
                f"fill ratio: {px_per_cm2:.1f} px^2/cm^2"
            )

        results = self.model(image, retina_masks=True, conf=self.conf_threshold, verbose=False)[0]

        empty_cols = ["item", "quantity", "weight_g", "kcal_per_100g", "calories"]
        if results.masks is None or len(results.masks) == 0:
            print("No food items detected.")
            return pd.DataFrame(columns=empty_cols)

        class_masks = defaultdict(list)
        for box, mask_obj in zip(results.boxes, results.masks):
            conf = float(box.conf[0])
            if conf < self.conf_threshold:
                continue
            class_id = int(box.cls[0])
            class_name = self.class_names.get(class_id, "unknown")

            mask = mask_obj.data[0].cpu().numpy()
            if mask.shape != (h, w):
                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            class_masks[class_name].append((mask > 0.5).astype(np.uint8))

        if not class_masks:
            print(f"No detections passed the confidence threshold ({self.conf_threshold}).")
            return pd.DataFrame(columns=empty_cols)

        rows = []
        total_calories = 0.0
        print("\n--- PlateCalc Diagnostic Engine ---")

        for class_name, masks in class_masks.items():
            combined_mask = np.zeros((h, w), dtype=np.uint8)
            for m in masks:
                combined_mask = np.maximum(combined_mask, m)

            weight_g, qty_desc, fdc_id = self._estimate_class_weight(
                class_name, combined_mask, len(masks), px_per_cm2
            )
            kcal_per_100g = self.get_kcal_per_100g(class_name, fdc_id)
            calories = (weight_g / 100.0) * kcal_per_100g
            total_calories += calories

            rows.append({
                "item": class_name, "quantity": qty_desc,
                "weight_g": round(weight_g, 1), "kcal_per_100g": round(kcal_per_100g, 1),
                "calories": round(calories, 1),
            })
            print(f"  {class_name:<22} | {qty_desc:<22} | {weight_g:>7.1f} g | {calories:>7.1f} kcal")

        print("-" * 70)
        print(f"TOTAL ESTIMATED MEAL CALORIES: {total_calories:.1f} kcal\n")

        df = pd.DataFrame(rows).sort_values("calories", ascending=False).reset_index(drop=True)

        if show_plot:
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            axes[0].imshow(cv2.cvtColor(results.plot(), cv2.COLOR_BGR2RGB))
            axes[0].set_title(f"Detections (total: {total_calories:.0f} kcal)")
            axes[0].axis("off")
            if df["calories"].sum() > 0:
                axes[1].pie(df["calories"], labels=df["item"], autopct="%1.0f%%", startangle=90)
                axes[1].set_title("Calorie contribution by item")
            plt.tight_layout()
            plt.show()

        return df

In [ ]:
# testing image
MODEL_PATH = "/content/drive/MyDrive/foodseg/final_run/weights/best.pt"
IMAGE_PATH = "/content/drive/MyDrive/foodseg/images.jpg" # for sample image ( within drive )

calc = PlateCalc(model_path=MODEL_PATH, usda_api_key=USDA_API_KEY)
breakdown_df = calc.analyze_plate(IMAGE_PATH)
breakdown_df
